# 07 — Security & Performance

Security and performance questions distinguish mid-level from senior candidates. These topics show you've built production systems.

---

## Table of Contents
1. Common Security Vulnerabilities
2. Input Validation & Sanitization
3. Authentication & Authorization
4. CORS
5. Security Headers (Helmet)
6. Rate Limiting
7. The Cluster Module
8. Caching Strategies
9. Memory Management & Leaks
10. Performance Optimization Tips
11. Interview Questions

---
## 1. Common Security Vulnerabilities (OWASP)

| Vulnerability | Description | Prevention |
|-------------|-----------|----------|
| **Injection** (SQL/NoSQL) | Untrusted data sent as part of a query | Parameterized queries, ORMs, input validation |
| **XSS** (Cross-Site Scripting) | Injecting malicious scripts | Output encoding, CSP headers, sanitize HTML |
| **CSRF** (Cross-Site Request Forgery) | Tricks user's browser into making requests | CSRF tokens, SameSite cookies |
| **Broken Auth** | Weak passwords, exposed tokens | bcrypt, JWT with short expiry, refresh tokens |
| **Sensitive Data Exposure** | Unencrypted data, leaked secrets | HTTPS, env vars, .gitignore, encryption at rest |
| **Security Misconfiguration** | Default configs, verbose errors | Helmet, disable X-Powered-By, env-specific configs |
| **Path Traversal** | Accessing files outside intended directory | Validate paths, use `path.resolve()`, avoid user input in paths |

In [ ]:
// SQL Injection — the classic vulnerability

// VULNERABLE (never do this!):
const userInput = "'; DROP TABLE users; --";
const badQuery = `SELECT * FROM users WHERE name = '${userInput}'`;
console.log('Vulnerable query:', badQuery);
// SELECT * FROM users WHERE name = ''; DROP TABLE users; --'

// SAFE: parameterized query (using placeholder)
const safeQuery = 'SELECT * FROM users WHERE name = $1';
const params = [userInput];
console.log('Safe query:', safeQuery, 'with params:', params);

// ALSO SAFE: using ORM (Sequelize, Prisma, etc.)
// const user = await User.findOne({ where: { name: userInput } });

In [ ]:
// NoSQL Injection (MongoDB)

// VULNERABLE — passing raw user input to MongoDB
// If req.body = { username: { "$gt": "" }, password: { "$gt": "" } }
// db.users.find({ username: { "$gt": "" } }) → returns all users!

// PREVENTION: validate types, use mongoose schema validation
function sanitizeMongoQuery(input) {
    if (typeof input === 'object' && input !== null) {
        for (const key of Object.keys(input)) {
            if (key.startsWith('$')) {
                delete input[key]; // Remove MongoDB operators
            }
        }
    }
    return input;
}

const malicious = { username: { '$gt': '' }, password: { '$gt': '' } };
console.log('Before:', JSON.stringify(malicious));
sanitizeMongoQuery(malicious.username);
sanitizeMongoQuery(malicious.password);
console.log('After:', JSON.stringify(malicious));

---
## 2. Input Validation & Sanitization

**Rule:** Never trust user input. Validate on the server, even if you validate on the client.

### Validation libraries:
- **Joi** — Schema-based, very popular
- **Zod** — TypeScript-first, growing fast
- **express-validator** — Express middleware
- **class-validator** — Decorator-based (NestJS)

```javascript
// Joi example
const Joi = require('joi');

const userSchema = Joi.object({
    name: Joi.string().min(2).max(50).required(),
    email: Joi.string().email().required(),
    age: Joi.number().integer().min(18).max(120),
    role: Joi.string().valid('user', 'admin').default('user'),
});

// In route handler:
const { error, value } = userSchema.validate(req.body);
if (error) return res.status(400).json({ error: error.details[0].message });
```

---
## 3. Authentication & Authorization

### Authentication (Who are you?) vs Authorization (What can you do?)

### JWT (JSON Web Token) — the most asked auth topic:
```
Structure: HEADER.PAYLOAD.SIGNATURE

Header:    { "alg": "HS256", "typ": "JWT" }
Payload:   { "sub": "user123", "role": "admin", "exp": 1700000000 }
Signature: HMACSHA256(base64(header) + "." + base64(payload), secret)
```

### Session vs JWT:
| Feature | Session-based | JWT |
|---------|-------------|-----|
| Storage | Server-side (memory/DB/Redis) | Client-side (cookie/localStorage) |
| Scalability | Needs shared session store | Stateless — scales easily |
| Revocation | Easy (delete from store) | Hard (need blocklist) |
| Size | Small cookie (session ID) | Larger (contains claims) |
| Best for | Traditional web apps | APIs, microservices, SPAs |

In [ ]:
const crypto = require('crypto');

// Password hashing (conceptual — use bcrypt in production)

// NEVER store plain text passwords!
// NEVER use MD5 or SHA for passwords (too fast, vulnerable to brute force)

// bcrypt example (pseudo-code — bcrypt is an npm package):
// const bcrypt = require('bcrypt');
// const hash = await bcrypt.hash('password123', 12);  // 12 salt rounds
// const isValid = await bcrypt.compare('password123', hash);

// Simple HMAC demonstration
const secret = 'my-secret-key';
const message = 'user-data';
const hmac = crypto.createHmac('sha256', secret).update(message).digest('hex');
console.log('HMAC:', hmac);

// JWT structure demonstration
function base64url(str) {
    return Buffer.from(str).toString('base64url');
}

const header = base64url(JSON.stringify({ alg: 'HS256', typ: 'JWT' }));
const payload = base64url(JSON.stringify({ sub: 'user123', role: 'admin' }));
const signature = crypto.createHmac('sha256', secret)
    .update(`${header}.${payload}`).digest('base64url');

console.log('JWT:', `${header}.${payload}.${signature}`);

---
## 4. CORS (Cross-Origin Resource Sharing)

CORS is a browser security mechanism. Browsers block requests from a different origin unless the server explicitly allows it.

```javascript
const cors = require('cors');

// Allow all origins (development only!)
app.use(cors());

// Production configuration
app.use(cors({
    origin: ['https://myapp.com', 'https://admin.myapp.com'],
    methods: ['GET', 'POST', 'PUT', 'DELETE'],
    allowedHeaders: ['Content-Type', 'Authorization'],
    credentials: true,    // Allow cookies
    maxAge: 86400,        // Preflight cache (24h)
}));
```

### Preflight requests:
For non-simple requests (PUT, DELETE, custom headers), the browser sends an OPTIONS request first to check if the server allows it. This is the "preflight."

---
## 5. Security Headers (Helmet)

```javascript
const helmet = require('helmet');
app.use(helmet()); // Sets ~15 security headers at once
```

### Key headers Helmet sets:
| Header | Purpose |
|--------|--------|
| `X-Content-Type-Options: nosniff` | Prevents MIME sniffing |
| `X-Frame-Options: DENY` | Prevents clickjacking |
| `Strict-Transport-Security` | Forces HTTPS |
| `Content-Security-Policy` | Controls allowed resource sources |
| `X-XSS-Protection` | XSS filter (legacy browsers) |
| Removes `X-Powered-By` | Hides Express fingerprint |

---
## 6. Rate Limiting

Prevents abuse, brute-force attacks, and DoS.

```javascript
const rateLimit = require('express-rate-limit');

const limiter = rateLimit({
    windowMs: 15 * 60 * 1000,  // 15 minutes
    max: 100,                   // 100 requests per window per IP
    message: { error: 'Too many requests, try again later' },
    standardHeaders: true,      // Return rate limit info in headers
});

app.use('/api', limiter);

// Stricter limiter for auth routes
const authLimiter = rateLimit({
    windowMs: 15 * 60 * 1000,
    max: 5,                    // Only 5 login attempts per 15 min
});
app.use('/api/auth/login', authLimiter);
```

---
## 7. The Cluster Module

Node.js is single-threaded, but modern servers have multiple CPU cores. The `cluster` module lets you fork multiple processes to use all cores.

In [ ]:
const cluster = require('cluster');
const os = require('os');

console.log('CPU cores available:', os.cpus().length);
console.log('Is primary/master:', cluster.isPrimary || cluster.isMaster);

// Cluster architecture:
// ┌─────────────────────────────────┐
// │        Primary Process          │
// │  (manages workers, no requests) │
// ├────────┬────────┬───────────────┤
// │Worker 1│Worker 2│   Worker N    │
// │(CPU 1) │(CPU 2) │   (CPU N)    │
// │ :3000  │ :3000  │    :3000     │
// └────────┴────────┴───────────────┘
//   All workers share the same port!

console.log(`\nCluster pattern:
if (cluster.isPrimary) {
    // Fork workers for each CPU
    for (let i = 0; i < os.cpus().length; i++) {
        cluster.fork();
    }
    cluster.on('exit', (worker) => {
        console.log('Worker died, spawning replacement');
        cluster.fork(); // Auto-restart
    });
} else {
    // Workers run the HTTP server
    app.listen(3000);
}`);

### Cluster vs PM2:
| Feature | `cluster` module | PM2 |
|---------|----------------|-----|
| Setup | Manual code | CLI tool |
| Auto-restart | Must implement | Built-in |
| Load balancing | Round-robin (default) | Round-robin + more |
| Monitoring | Manual | Built-in dashboard |
| Log management | Manual | Built-in |
| Zero-downtime restart | Must implement | `pm2 reload` |

```bash
# PM2 commands
pm2 start app.js -i max     # Start with max CPU cores
pm2 reload app              # Zero-downtime restart
pm2 monit                   # Live monitoring
pm2 logs                    # View logs
pm2 save                    # Save process list
pm2 startup                 # Auto-start on boot
```

---
## 8. Caching Strategies

| Level | Where | What | Tool |
|-------|-------|------|------|
| **Application** | In-process memory | Computed results, config | `node-cache`, Map |
| **Distributed** | External cache server | Session, API responses | Redis, Memcached |
| **HTTP** | Browser/CDN | Static assets, API responses | Cache-Control headers |
| **Database** | Query cache | Frequent queries | DB-specific, Redis |

### Redis caching pattern:
```javascript
async function getUserWithCache(userId) {
    const cacheKey = `user:${userId}`;
    
    // Check cache first
    const cached = await redis.get(cacheKey);
    if (cached) return JSON.parse(cached);
    
    // Cache miss — fetch from DB
    const user = await db.users.findById(userId);
    
    // Store in cache with TTL
    await redis.set(cacheKey, JSON.stringify(user), 'EX', 3600); // 1 hour
    
    return user;
}
```

### Cache invalidation strategies:
- **TTL (Time to Live)** — expires after N seconds
- **Write-through** — update cache when DB updates
- **Cache-aside** — app manages cache reads/writes
- **Write-behind** — batch writes to DB

---
## 9. Memory Management & Leaks

In [ ]:
// Monitoring memory usage
const used = process.memoryUsage();
console.log('Memory Usage:');
for (const [key, value] of Object.entries(used)) {
    console.log(`  ${key}: ${(value / 1024 / 1024).toFixed(2)} MB`);
}

// Key metrics:
// heapTotal — total heap allocated by V8
// heapUsed  — actual memory used by JS objects
// rss       — Resident Set Size (total allocated by OS)
// external  — C++ objects bound to JS (Buffers)
// arrayBuffers — ArrayBuffer and SharedArrayBuffer memory

In [ ]:
// Common memory leak patterns

// 1. Global variables that grow indefinitely
// const cache = {}; // Never cleared → leak!

// 2. Event listeners not removed
// Especially in long-running servers:
// emitter.on('data', handler); // Added on every request → leak!

// 3. Closures holding references
function createLeak() {
    const bigData = new Array(1000000).fill('x');
    return () => bigData.length; // Closure keeps bigData alive
}

// 4. Forgotten timers
// const interval = setInterval(() => { /* work */ }, 1000);
// Must call clearInterval(interval) when done

// Fix: use WeakMap/WeakSet for object caches
const weakCache = new WeakMap();
let obj = { data: 'important' };
weakCache.set(obj, 'metadata');
obj = null; // weakCache entry is now eligible for GC!

console.log('Common leak sources: global caches, event listeners, closures, timers');

---
## 10. Performance Optimization Tips

### Quick wins:
1. **Use `async/await`** — don't block the event loop
2. **Enable gzip compression** — `compression` middleware
3. **Use streams** for large data instead of `readFileSync`
4. **Connection pooling** for databases
5. **Cache frequently accessed data** (Redis)
6. **Use reverse proxy** (Nginx) for static files and SSL termination
7. **Set `NODE_ENV=production`** — Express caches views, less verbose errors
8. **Use `cluster` or PM2** to use all CPU cores

### Monitoring & profiling:
```bash
node --prof app.js           # V8 profiler
node --inspect app.js        # Chrome DevTools profiler
clinic doctor -- node app.js # clinic.js (npm package)
```

### Event loop lag detection:
```javascript
// If this takes much longer than the interval, event loop is blocked
let lastCheck = Date.now();
setInterval(() => {
    const now = Date.now();
    const lag = now - lastCheck - 1000; // expected 1000ms
    if (lag > 100) console.warn(`Event loop lag: ${lag}ms`);
    lastCheck = now;
}, 1000);
```

---
## 11. Interview Questions & Answers

### Q1: How do you prevent SQL/NoSQL injection?
**A:** Use parameterized queries (never string concatenation), ORMs like Sequelize/Prisma, input validation (Joi/Zod), and type checking for MongoDB (reject objects with `$` operators).

### Q2: What is JWT and how does it work?
**A:** JWT is a signed token with three base64-encoded parts: header (algorithm), payload (claims like userId, role, expiry), and signature (HMAC of header+payload using a secret). The server verifies the signature without needing a database lookup, making it stateless.

### Q3: How do you scale a Node.js application?
**A:** Vertically: cluster module or PM2 to use all CPU cores. Horizontally: multiple servers behind a load balancer (Nginx, AWS ALB). Plus: caching (Redis), database read replicas, CDN for static assets, message queues for async processing.

### Q4: What is CORS and why is it needed?
**A:** CORS is a browser security mechanism that restricts web pages from making requests to a different origin (domain, port, or protocol). It's needed because without it, malicious sites could make requests to your API using the user's cookies. Servers must explicitly allow cross-origin access via response headers.

### Q5: How do you detect and fix memory leaks in Node.js?
**A:** Monitor `process.memoryUsage()` over time. Use `--inspect` with Chrome DevTools to take heap snapshots. Common causes: global variables, unremoved event listeners, closures holding references, growing caches without eviction, forgotten timers. Fix with WeakMap/WeakSet, proper cleanup, TTL caches.

### Q6: Session-based auth vs JWT — when to use each?
**A:** Sessions are better for traditional web apps (easy revocation, smaller cookies). JWT is better for APIs and microservices (stateless, no shared session store needed). For SPAs, JWT with refresh token rotation is common. Always use httpOnly cookies to store tokens to prevent XSS.